In [ ]:
# Celula 1 - Setup Colab resiliente v2 (clone/zip + token opcional + validacao de import)
import os
import sys
import shutil
import subprocess
import urllib.request
import zipfile
import io
from pathlib import Path

SETUP_VERSION = "2026-04-29-v2"
print("SETUP_VERSION:", SETUP_VERSION)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = Path("/content/TCC")
BRANCH = "update"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()  # opcional para repo privado

def _run(cmd, *, capture=False):
    print("Executando:", " ".join(map(str, cmd)))
    return subprocess.run(cmd, text=True, capture_output=capture)

def _repo_http_base(repo_url: str) -> str:
    return repo_url[:-4] if repo_url.endswith(".git") else repo_url

def _repo_slug(repo_url: str) -> str:
    base = _repo_http_base(repo_url).rstrip("/")
    return "/".join(base.split("/")[-2:])  # owner/repo

def _auth_clone_url(repo_url: str) -> str:
    if not GITHUB_TOKEN:
        return repo_url
    base = _repo_http_base(repo_url)
    return base.replace("https://", f"https://{GITHUB_TOKEN}@") + ".git"

def _try_clone(repo_url: str, repo_dir: Path, branch: str | None):
    url = _auth_clone_url(repo_url)
    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [url, str(repo_dir)]
    proc = _run(cmd, capture=True)
    ok = proc.returncode == 0
    if not ok:
        print("[clone] falhou com codigo", proc.returncode)
        if proc.stderr:
            print(proc.stderr[-2500:])
    return ok

def _download_zip_codeload(repo_url: str, repo_dir: Path, branch: str):
    slug = _repo_slug(repo_url)
    zip_url = f"https://codeload.github.com/{slug}/zip/refs/heads/{branch}"
    print("Tentando ZIP:", zip_url)

    req = urllib.request.Request(zip_url)
    if GITHUB_TOKEN:
        req.add_header("Authorization", f"token {GITHUB_TOKEN}")

    data = urllib.request.urlopen(req).read()
    zf = zipfile.ZipFile(io.BytesIO(data))
    zf.extractall(repo_dir.parent)

    expected_prefix = f"{slug.split('/')[-1]}-{branch}"
    extracted = None
    for p in repo_dir.parent.iterdir():
        if p.is_dir() and p.name.startswith(expected_prefix):
            extracted = p
            break
    if extracted is None:
        raise RuntimeError(f"Pasta extraida nao encontrada para prefixo {expected_prefix}")
    extracted.rename(repo_dir)

def _obtain_repo(repo_url: str, repo_dir: Path, branch: str):
    errors = []

    if _try_clone(repo_url, repo_dir, branch):
        return
    errors.append(f"clone-branch:{branch}")

    if _try_clone(repo_url, repo_dir, None):
        return
    errors.append("clone-default")

    for b in [branch, "main", "master"]:
        if b in {e.replace('clone-branch:', '') for e in []}:
            pass
        try:
            _download_zip_codeload(repo_url, repo_dir, b)
            print(f"Repositorio obtido via ZIP da branch '{b}'.")
            return
        except Exception as exc:
            print(f"ZIP da branch '{b}' falhou: {exc}")
            errors.append(f"zip:{b}")

    raise RuntimeError("Nao foi possivel obter repositorio. Tentativas: " + ", ".join(errors))

def _ensure_importable(repo_dir: Path):
    src = repo_dir / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

    try:
        import nvs_benchmark  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_dir)], check=True)
        import nvs_benchmark  # noqa: F401

    probe = [
        sys.executable,
        "-c",
        "import nvs_benchmark,sys; print('nvs_benchmark OK em', sys.executable)",
    ]
    subprocess.run(probe, check=True)

if IN_COLAB:
    if REPO_DIR.exists():
        print("Removendo pasta existente:", REPO_DIR)
        shutil.rmtree(REPO_DIR)

    _obtain_repo(REPO_URL, REPO_DIR, BRANCH)

    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

    req = REPO_DIR / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)

    _ensure_importable(REPO_DIR)
    print("Setup concluido. Modulo nvs_benchmark importavel.")
else:
    print("Nao detectado Colab. Ajuste os caminhos para execucao local.")

In [ ]:
# Celula 2 - Parametros da matriz completa
RUN_ID = "colab_full_matrix"
PRESET = "quick"  # smoke, quick, preview, standard, full
STRICT_RESULTS = True
MIN_REQUIRED_PAIRS = 1
GENERATE_PDF = False

# Controle de escopo (None = todos os disponiveis)
ONLY_DATASETS = None  # ex.: ["blender_synthetic"]
ONLY_METHODS = None   # ex.: ["nerf_static", "gs_static"]

# Caminhos base
ARTIFACTS_DIR = Path("./artifacts")
METRICS_DIR = ARTIFACTS_DIR / "metrics"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
LOG_DIR = Path("./logs")

# Snapshot consolidado final
SNAPSHOT_FILE = METRICS_DIR / f"{RUN_ID}.json"
REPORT_NAME = f"{RUN_ID}_report"

# Opcional: tentar instalar datasets via catalogo antes de rodar
RUN_DATASET_INSTALL = False

In [ ]:
# Celula 3 - Validacoes de ambiente e utilitarios
import shutil
import traceback
from glob import glob

import torch

REPO_DIR = Path.cwd()
print("Diretorio atual:", REPO_DIR)
print("Python:", sys.executable)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if IN_COLAB:
    usage = shutil.disk_usage("/content")
    free_gb = usage.free / (1024**3)
    print(f"Espaco livre em /content: {free_gb:.2f} GB")

def run_cmd(cmd: list[str], check=True):
    print("\n$", " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=True)

def ensure_imports():
    src = REPO_DIR / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
    try:
        import nvs_benchmark  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
        import nvs_benchmark  # noqa: F401

ensure_imports()
print("Imports principais validados.")

In [ ]:
# Celula 4 - Instalacao opcional de datasets via catalogo
METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if RUN_DATASET_INSTALL:
    cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "install",
        "--catalog-file", "./configs/install_catalog.json",
        "--only", "datasets",
        "--execute",
    ]
    proc = run_cmd(cmd, check=False)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        print("Aviso: instalacao de datasets falhou; o notebook tentara usar o que ja existir.")
else:
    print("RUN_DATASET_INSTALL=False. Pulando instalacao de datasets.")

In [ ]:
# Celula 5 - Descoberta de datasets e metodos
ensure_imports()

from nvs_benchmark.data import SUPPORTED_DATASETS
from nvs_benchmark.methods import build_registry_with_all_methods

dataset_base_candidates = {
    "blender_synthetic": [
        Path("./data/blender_synthetic/nerf_synthetic/lego"),
        Path("./data/blender_synthetic/lego"),
    ],
    "d_nerf": [
        Path("./data/d_nerf"),
    ],
    "mipnerf360": [
        Path("./data/mipnerf360"),
    ],
    "tanks_and_temples": [
        Path("./data/tanks_and_temples"),
    ],
    "custom": [
        Path("./data/custom"),
    ],
}

def choose_scene_root(dataset_name: str) -> Path | None:
    candidates = dataset_base_candidates.get(dataset_name, [])
    for base in candidates:
        if (base / "transforms_train.json").exists():
            return base
        hits = list(base.rglob("transforms_train.json")) if base.exists() else []
        if hits:
            return hits[0].parent
    return None

all_datasets = list(SUPPORTED_DATASETS)
selected_datasets = ONLY_DATASETS if ONLY_DATASETS else all_datasets
selected_datasets = [d for d in selected_datasets if d in all_datasets]

registry = build_registry_with_all_methods()
if hasattr(registry, "list_ids"):
    all_methods = registry.list_ids()
elif hasattr(registry, "keys"):
    all_methods = list(registry.keys())
else:
    raise RuntimeError("Nao foi possivel listar metodos no registry.")

selected_methods = ONLY_METHODS if ONLY_METHODS else all_methods
selected_methods = [m for m in selected_methods if m in all_methods]

print("Datasets suportados:", all_datasets)
print("Datasets selecionados:", selected_datasets)
print("Metodos selecionados:", selected_methods)

if not selected_methods:
    raise RuntimeError("Nenhum metodo encontrado no registry. Verifique dependencias externas.")

In [ ]:
# Celula 6 - Construir plano de execucao method x dataset
matrix_plan = []
for dataset_name in selected_datasets:
    root = choose_scene_root(dataset_name)
    if root is None:
        print(f"[SKIP] Dataset sem root valido: {dataset_name}")
        continue
    for method_name in selected_methods:
        matrix_plan.append({
            "dataset": dataset_name,
            "root": str(root),
            "method": method_name,
            "split": "train",
        })

print(f"Total de combinacoes planejadas: {len(matrix_plan)}")
for i, item in enumerate(matrix_plan[:10], 1):
    print(f"{i:02d}. {item['dataset']} x {item['method']} -> {item['root']}")
if len(matrix_plan) > 10:
    print("... (mostrando apenas as 10 primeiras)")

if not matrix_plan:
    raise RuntimeError("Nenhuma combinacao valida encontrada. Verifique os datasets em ./data.")

In [ ]:
# Celula 7 - Executar matriz completa
run_results = []

for idx, item in enumerate(matrix_plan, 1):
    ds = item["dataset"]
    method = item["method"]
    root = item["root"]
    pair_id = f"{ds}__{method}"
    pair_snapshot = METRICS_DIR / f"{RUN_ID}_{pair_id}.json"

    cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "method-run",
        "--method", method,
        "--dataset", ds,
        "--root", root,
        "--split", item["split"],
        "--preset", PRESET,
        "--output-dir", str(ARTIFACTS_DIR),
        "--log-dir", str(LOG_DIR),
        "--compute-metrics",
        "--snapshot-file", str(pair_snapshot),
        "--append-snapshot",
    ]
    if STRICT_RESULTS:
        cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

    print(f"\n[{idx}/{len(matrix_plan)}] Rodando: {ds} x {method}")
    proc = run_cmd(cmd, check=False)
    ok = proc.returncode == 0
    if proc.stdout:
        print(proc.stdout[-1500:])
    if not ok and proc.stderr:
        print(proc.stderr[-2000:])

    run_results.append({
        "dataset": ds,
        "method": method,
        "root": root,
        "snapshot_file": str(pair_snapshot),
        "ok": ok,
        "returncode": proc.returncode,
    })

ok_count = sum(1 for r in run_results if r["ok"])
print(f"Concluido: {ok_count}/{len(run_results)} combinacoes com sucesso")

In [ ]:
# Celula 8 - Consolidar snapshots individuais em snapshot final
consolidated = {}

for item in run_results:
    snap = Path(item["snapshot_file"])
    if not item["ok"] or not snap.exists():
        continue
    try:
        payload = json.loads(snap.read_text(encoding="utf-8"))
        if isinstance(payload, dict):
            consolidated.update(payload)
    except Exception:
        print(f"Aviso: snapshot invalido ignorado: {snap}")

SNAPSHOT_FILE.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_FILE.write_text(json.dumps(consolidated, indent=2, ensure_ascii=False), encoding="utf-8")
print("Snapshot consolidado:", SNAPSHOT_FILE)
print("Entradas consolidadas:", len(consolidated))

In [ ]:
# Celula 9 - Resumo da execucao e diagnostico
from collections import Counter

status_counter = Counter("ok" if r["ok"] else "fail" for r in run_results)
print("Resumo:", dict(status_counter))

failed = [r for r in run_results if not r["ok"]]
if failed:
    print("\nCombinacoes com falha:")
    for f in failed[:20]:
        print(f"- {f['dataset']} x {f['method']} (code={f['returncode']})")
    if len(failed) > 20:
        print("... (mostrando apenas as 20 primeiras)")

if not consolidated:
    raise RuntimeError("Snapshot consolidado vazio. Nao ha resultados para gerar relatorio.")

In [ ]:
# Celula 10 - Gerar relatorio consolidado
report_cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", str(SNAPSHOT_FILE),
    "--output-dir", str(REPORTS_DIR),
    "--report-name", REPORT_NAME,
    "--log-dir", str(LOG_DIR),
]
if not GENERATE_PDF:
    report_cmd.append("--no-pdf")
if STRICT_RESULTS:
    report_cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"] )

proc = run_cmd(report_cmd, check=False)
if proc.stdout:
    print(proc.stdout[-2000:])
if proc.returncode != 0:
    if proc.stderr:
        print(proc.stderr[-2000:])
    raise RuntimeError("Falha ao gerar relatorio consolidado.")

REPORT_HTML = REPORTS_DIR / f"{REPORT_NAME}.html"
print("Relatorio HTML:", REPORT_HTML)

In [ ]:
# Celula 11 - Exibir relatorio no notebook
from IPython.display import IFrame, display

if REPORT_HTML.exists():
    display(IFrame(src=str(REPORT_HTML), width=1200, height=720))
else:
    print("Relatorio HTML nao encontrado:", REPORT_HTML)

In [ ]:
# Celula 12 - Compactar artefatos e baixar no Colab
import pathlib

archive_base = "/content/nvs_benchmark_full_matrix_artifacts"
archive_file = shutil.make_archive(archive_base, "zip", str(REPO_DIR), "artifacts")
print("Arquivo gerado:", archive_file)

if IN_COLAB and pathlib.Path(archive_file).exists():
    colab_files_mod = __import__("google.colab", fromlist=["files"])
    files = getattr(colab_files_mod, "files")
    files.download(archive_file)

In [ ]:
# Celula 13 - Backup opcional para Google Drive
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/NVS_Benchmark_Full_Matrix")

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")

    target_artifacts = DRIVE_OUTPUT_DIR / "artifacts"
    if target_artifacts.exists():
        shutil.rmtree(target_artifacts)
    shutil.copytree(ARTIFACTS_DIR, target_artifacts)
    print("Backup concluido em:", target_artifacts)
else:
    print("Backup no Drive desabilitado (USE_GOOGLE_DRIVE=False).")

In [ ]:
# Celula 14 - Resumo final
total = len(run_results)
ok = sum(1 for r in run_results if r["ok"])
fail = total - ok

print("=" * 70)
print("NVS Benchmark - Full Matrix (Colab)")
print("=" * 70)
print("Run ID:", RUN_ID)
print("Preset:", PRESET)
print(f"Combinacoes: {ok}/{total} sucesso, {fail} falha")
print("Snapshot consolidado:", SNAPSHOT_FILE)
print("Relatorio HTML:", REPORT_HTML)
print("Logs:", LOG_DIR)
print("=" * 70)

if fail > 0:
    print("Sugestao: execute novamente apenas combinacoes com falha usando ONLY_DATASETS/ONLY_METHODS.")